## HumanInTheLoopMiddleware
- 用于调用工具前,让用户决定是否调用工具及修改参数


In [4]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from langchain.tools import tool

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    profile={"max_input_tokens": 128_000},
    extra_body = {"thinking": None},
)

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """
    通过邮件ID读取内容的伪函数
    """
    return f"邮件ID：{email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """
    发送邮件伪函数
    """
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent = create_agent(
    model = model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),#后面会介绍
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"get_weather": False,#是否中断
                          "get_news": True,
                          #3种决策类型:edit,approve,reject直接使用字典作为 InterruptOnConfig
                          "read_email_tool": {"allowed_decisions":["edit"]},
                          "send_email_tool": {"allowed_decisions":["approve","reject"],
                                              "description":"发送邮件失败"}},#Interrupt消息描述,优先级高
            description_prefix="中断了!"#Interrupt消息描述,优先级低,未自定义时使用(True)
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

message =[HumanMessage(content="请帮我查询今天北京的天气"
"查询今日新闻"
"查看ID为 'sk2131421' 的邮件内容，"
"向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
"同时做这四件事")]

response = agent.invoke({
    "messages": message
        },
    config=config
)

rprint(response)



{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='63a5d5d2-9f59-497a-8fca-6177a01d8f3b'
        ),
        AIMessage(
            content='我来同时执行这四件事：查询北京天气、查看今日新闻、读取指定邮件、发送邮件。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': "The user wants me to do four things simultaneously:\n1. Query today's weather
in Beijing\n2. Query today's news\n3. Read email with ID 'sk2131421'\n4. Send email to 15641685664@qq.com with 
subject '哈哈哈' and body '你好啊'\n\nThese are all independent calls, so I can make them all at once."
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 266,
                    'prompt_tokens': 594,
                    'total_tokens': 860,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 77,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 82
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'a0f20526-a3f1-4300-a37b-320b3de09d17',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a01032-98da-75c1-93c5-0ddca6ad2926-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_HV6Oar7XptA4mQu6BTMA5343',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_MLajsxO8tjOmVFQ35KHa6783', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_CS0krWK57KhEK6CDIGud4927',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_jXQpSFtW5AQMyl8CAdZh9625',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 594,
                'output_tokens': 266,
                'total_tokens': 860,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {'reasoning': 77}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {'name': 'get_news', 'args': {}, 'description': '中断了!\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'read_email_tool',
                        'args': {'email_id': 'sk2131421'},
                        'description': "中断了!\n\nTool: read_email_tool\nArgs: {'email_id': 'sk2131421'}"
                    },
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件失败'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'read_em

In [5]:
from langgraph.types import Command

interrupt = response.get("__interrupt__",[])
rupts = interrupt[0].value["action_requests"]

decisions = []

#用户决策
news_config = {
    "type" : "approve"
}

read_email_tool_config = {
    "type" : "edit",
    "edited_action" : {
        "name" : "read_email_tool",
        "args" : {"email_id" : "sk2131222"}
    }
}

send_email_tool_config = {
    "type" : "approve"
}

#根据rupts,生成decisions,顺序需与rupts一致
for i in range(len(rupts)):
    if rupts[i]["name"] == "get_news":
        decisions.append(news_config)
    elif rupts[i]["name"] == "read_email_tool":
        decisions.append(read_email_tool_config)
    elif rupts[i]["name"] == "send_email_tool":
        decisions.append(send_email_tool_config)

res = agent.invoke(
    #继续执行,并传递decisions
    Command(resume={"decisions": decisions}),
    config=config
)

rprint(res)

>>> 真的执行发送邮件工具了


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='63a5d5d2-9f59-497a-8fca-6177a01d8f3b'
        ),
        AIMessage(
            content='我来同时执行这四件事：查询北京天气、查看今日新闻、读取指定邮件、发送邮件。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': "The user wants me to do four things simultaneously:\n1. Query today's weather
in Beijing\n2. Query today's news\n3. Read email with ID 'sk2131421'\n4. Send email to 15641685664@qq.com with 
subject '哈哈哈' and body '你好啊'\n\nThese are all independent calls, so I can make them all at once."
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 266,
                    'prompt_tokens': 594,
                    'total_tokens': 860,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 77,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 82
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'a0f20526-a3f1-4300-a37b-320b3de09d17',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a01032-98da-75c1-93c5-0ddca6ad2926-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_HV6Oar7XptA4mQu6BTMA5343',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_MLajsxO8tjOmVFQ35KHa6783', 'type': 'tool_call'},
                {
                    'type': 'tool_call',
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131222'},
                    'id': 'call_02_CS0krWK57KhEK6CDIGud4927'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_jXQpSFtW5AQMyl8CAdZh9625',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 594,
                'output_tokens': 266,
                'total_tokens': 860,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {'reasoning': 77}
            }
        ),
        ToolMessage(
            content='北京今天天气不错',
            name='get_weather',
            id='882810e5-db57-48cd-a1f3-f0afb3ed4ef0',
            tool_call_id='call_00_HV6Oar7XptA4mQu6BTMA5343'
        ),
        ToolMessage(
            content='中方三艘油轮通过霍尔木兹海峡',
            name='get_news',
            id='97a721a6-8cd9-4cb2-bcd0-85354c738c53',
            tool_call_id='call_01_MLajsxO8tjOmVFQ35KHa6783'
        ),
        ToolMessage(
            content='邮件ID：sk2131222\n是空的',
            name='read_email_tool',
            id='5039d89f-81ef-4788-9b69-eeb76d4a0973',
            tool_call_id='call_02_CS0krWK57KhEK6CDIGud4927'
        ),
        ToolMessage(
            content='发送给 15641685664@qq.com 的邮件标题是：哈哈哈，内容：你好啊',
            name='send_email_tool',
            id='d1ce7130-6929-45a4-a095-33fd0d90e231',
            tool_call_id='call_03_jXQpSFtW5AQMyl8CAdZh9625'
        ),
        AIMessag